# R-AIME Explainability and Visualization Analysis

This notebook demonstrates the comprehensive analysis and visualization capabilities of the R-AIME (Reinforcement-Augmented Interpretable Multimodal Emotion) framework.

## Overview

The R-AIME framework consists of three main agents:
1. **Fusion Agent**: Combines multimodal features with importance weighting
2. **Decision Agent**: Provides calibrated predictions and confidence estimates
3. **Explainability Agent**: Offers feature attribution and concept activation analysis

## Setup

In [ ]:
import sys
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Add project root to path
project_root = Path('../')
sys.path.append(str(project_root))

# Import our modules
from raime_analysis import RAIMEVisualizer, run_complete_raime_analysis
from models.model import IEMOCAPMemoCMTModel, load_config
from data.iemocap import create_dataloaders
from utils.common import set_seed
from transformers import AutoTokenizer, Wav2Vec2Processor

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("Setup completed!")

## 1. Load Model and Data

First, let's load a trained model and prepare the test data for analysis.

In [ ]:
def load_model_and_data(config_path: str, checkpoint_path: str, batch_size: int = 8):
    """Load model and create dataloader for analysis"""
    print("Loading model and data...")
    
    # Load configuration
    config = load_config(config_path)
    
    # Override batch size for analysis
    config['training']['batch_size'] = batch_size
    
    # Initialize tokenizer and processor
    text_config = config.get('model', {}).get('text', {})
    audio_config = config.get('model', {}).get('audio', {})
    
    tokenizer = AutoTokenizer.from_pretrained(
        text_config.get('model_name', 'j-hartmann/emotion-english-distilroberta-base')
    )
    
    processor = Wav2Vec2Processor.from_pretrained(
        audio_config.get('model_name', 'facebook/wav2vec2-base-960h')
    )
    
    # Load model from checkpoint
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = IEMOCAPMemoCMTModel(config, tokenizer, processor)
    
    # Load checkpoint
    checkpoint = torch.load(checkpoint_path, map_location=device)
    if 'state_dict' in checkpoint:
        state_dict = checkpoint['state_dict']
    else:
        state_dict = checkpoint
    
    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()
    
    print(f"Model loaded on {device}")
    
    # Create dataloader
    dataset_config = config.get('dataset', {})
    root_path = dataset_config.get('root_path', '../iemocap_dataset')
    
    _, _, test_loader = create_dataloaders(
        root_path=root_path,
        tokenizer=tokenizer,
        processor=processor,
        batch_size=batch_size,
        num_workers=2,
        sample_rate=dataset_config.get('sample_rate', 16000)
    )
    
    print("Data loader created")
    
    return model, test_loader, config, device

# Configuration - Update these paths for your setup
CONFIG_PATH = 'configs/iemocap_memocmt.yaml'
CHECKPOINT_PATH = 'checkpoints/best_model.pth'  # Update with your checkpoint path
BATCH_SIZE = 4

# Load model and data
try:
    model, test_loader, config, device = load_model_and_data(
        CONFIG_PATH, CHECKPOINT_PATH, BATCH_SIZE
    )
    emotion_labels = config.get('dataset', {}).get('emotion_labels', {})
    print(f"Emotion classes: {list(emotion_labels.keys())}")
except Exception as e:
    print(f"Error loading model: {e}")
    print("Please update the CONFIG_PATH and CHECKPOINT_PATH variables with valid paths.")
    model, test_loader, config, device, emotion_labels = None, None, None, None, None

## 2. Fusion Agent Analysis

The Fusion Agent combines multimodal features with importance weighting and attention mechanisms.

In [ ]:
if model is not None:
    print("Analyzing Fusion Agent...")
    
    # Initialize visualizer
    visualizer = RAIMEVisualizer(emotion_labels)
    
    # Run fusion agent analysis
    fusion_results = visualizer.analyze_fusion_agent(
        model, test_loader, device, num_samples=30
    )
    
    print("Fusion Agent Results:")
    print(f"   Samples analyzed: {len(fusion_results['predictions'])}")
    print(f"   Accuracy: {np.mean(fusion_results['predictions'] == fusion_results['labels']):.3f}")
    
    if fusion_results['feature_importance'].size > 0:
        print(f"   Mean feature importance: {np.mean(fusion_results['feature_importance']):.3f}")
        print(f"   Feature importance std: {np.std(fusion_results['feature_importance']):.3f}")
    
    print("\nFusion Agent analysis completed!")
    print("Check the 'raime_analysis' folder for generated visualizations.")
else:
    print("Model not loaded. Please check the configuration and try again.")

## 3. Decision Agent Analysis

The Decision Agent provides calibrated predictions and confidence estimates.

In [ ]:
if model is not None:
    print("Analyzing Decision Agent...")
    
    # Run decision agent analysis
    decision_results = visualizer.analyze_decision_agent(
        model, test_loader, device, num_samples=30
    )
    
    predictions = decision_results['predictions']
    labels = decision_results['labels']
    confidences = decision_results['confidences'].flatten()
    correct_mask = predictions == labels
    
    print("Decision Agent Results:")
    print(f"   Samples analyzed: {len(predictions)}")
    print(f"   Accuracy: {np.mean(correct_mask):.3f}")
    print(f"   Mean confidence: {np.mean(confidences):.3f}")
    print(f"   Confidence correct predictions: {np.mean(confidences[correct_mask]):.3f}")
    print(f"   Confidence incorrect predictions: {np.mean(confidences[~correct_mask]):.3f}")
    
    print("\n Decision Agent analysis completed!")
else:
    print(" Model not loaded. Please check the configuration and try again.")

## 4. Explainability Agent Analysis

The Explainability Agent provides feature attribution and concept activation analysis.

In [ ]:
if model is not None:
    print("Analyzing Explainability Agent...")
    
    # Run explainability agent analysis
    explainability_results = visualizer.analyze_explainability_agent(
        model, test_loader, device, num_samples=15
    )
    
    print("Explainability Agent Results:")
    print(f"   Samples analyzed: {len(explainability_results['predictions'])}")
    
    if explainability_results['feature_attributions'].size > 0:
        attributions = explainability_results['feature_attributions']
        print(f"   Feature attributions shape: {attributions.shape}")
        print(f"   Mean attribution magnitude: {np.mean(np.abs(attributions)):.3f}")
    
    if explainability_results['concept_activations'].size > 0:
        concepts = explainability_results['concept_activations']
        print(f"   Concept activations shape: {concepts.shape}")
    
    print("\nExplainability Agent analysis completed!")
else:
    print("Model not loaded. Please check the configuration and try again.")

## 5. Attention Weights Analysis

Analyze the attention patterns in the MemoCMT transformer layers.

In [ ]:
if model is not None:
    print("Analyzing Attention Weights...")
    
    # Run attention analysis
    attention_results = visualizer.analyze_attention_weights(
        model, test_loader, device, num_samples=10
    )
    
    print("Attention Analysis Results:")
    print(f"   Samples analyzed: {len(attention_results['predictions'])}")
    
    attention_types = ['text_self_attention', 'audio_self_attention', 
                      'text_cross_attention', 'audio_cross_attention']
    
    for attn_type in attention_types:
        if attention_results[attn_type].size > 0:
            print(f"   {attn_type}: {attention_results[attn_type].shape}")
    
    if attention_results['memory_states'].size > 0:
        print(f"   Memory states: {attention_results['memory_states'].shape}")
    
    print("\nAttention analysis completed!")
else:
    print("Model not loaded. Please check the configuration and try again.")

## 6. Comprehensive Analysis Report

Generate a comprehensive analysis report with all findings.

In [ ]:
if model is not None:
    print("Generating Comprehensive Analysis Report...")
    
    # Generate comprehensive report
    report = visualizer.generate_comprehensive_report(
        fusion_results, decision_results, 
        explainability_results, attention_results
    )
    
    print("Summary Statistics:")
    for key, value in report['summary'].items():
        print(f"   {key}: {value:.3f}")
    
    print("\n Performance Metrics:")
    for key, value in report['metrics'].items():
        print(f"   {key}: {value:.3f}")
    
    print("\nKey Insights:")
    for i, insight in enumerate(report['insights'], 1):
        print(f"   {i}. {insight}")
    
    print("\n Comprehensive analysis completed!")
    print(f" All results saved to: {visualizer.output_dir}")
else:
    print(" Model not loaded. Cannot generate report.")

## 7. Interactive Visualization Examples

Let's create some interactive visualizations to explore the results in more detail.

In [ ]:
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

def create_interactive_confidence_plot(confidences, predictions, labels, emotion_labels):
    """Create interactive confidence visualization"""
    
    correct_mask = predictions == labels
    class_names = [None] * len(emotion_labels)
    for emotion, idx in emotion_labels.items():
        class_names[idx] = emotion.capitalize()
    
    # Create scatter plot
    fig = go.Figure()
    
    # Correct predictions
    fig.add_trace(go.Scatter(
        x=np.where(correct_mask)[0],
        y=confidences[correct_mask],
        mode='markers',
        name='Correct',
        marker=dict(color='green', size=8),
        text=[f'Predicted: {class_names[pred]}<br>True: {class_names[label]}<br>Confidence: {conf:.3f}' 
              for pred, label, conf in zip(predictions[correct_mask], labels[correct_mask], confidences[correct_mask])]
    ))
    
    # Incorrect predictions
    fig.add_trace(go.Scatter(
        x=np.where(~correct_mask)[0],
        y=confidences[~correct_mask],
        mode='markers',
        name='Incorrect',
        marker=dict(color='red', size=8),
        text=[f'Predicted: {class_names[pred]}<br>True: {class_names[label]}<br>Confidence: {conf:.3f}' 
              for pred, label, conf in zip(predictions[~correct_mask], labels[~correct_mask], confidences[~correct_mask])]
    ))
    
    fig.update_layout(
        title='Interactive Confidence Analysis',
        xaxis_title='Sample Index',
        yaxis_title='Confidence Score',
        hovermode='closest'
    )
    
    return fig

# Create interactive visualization if we have results
if 'decision_results' in locals() and decision_results is not None:
    fig = create_interactive_confidence_plot(
        decision_results['confidences'].flatten(),
        decision_results['predictions'],
        decision_results['labels'],
        emotion_labels
    )
    fig.show()
    print("Interactive confidence plot created!")
else:
    print("No decision results available for interactive visualization.")

## 8. Summary and Next Steps

This notebook demonstrated the comprehensive R-AIME analysis capabilities:

### What We Analyzed:
1. **Fusion Agent**: Feature importance and attention patterns
2. **Decision Agent**: Confidence calibration and prediction reliability
3. **Explainability Agent**: Feature attribution and concept activations
4. **Attention Weights**: Cross-modal attention and memory states

### Generated Outputs:
- Static visualizations (PNG files) in the `raime_analysis/` folder
- Interactive plots for detailed exploration
- Comprehensive analysis report (JSON)
- Performance metrics and insights

### Next Steps:
1. **Customize Analysis**: Modify the `num_samples` parameter for different analysis depths
2. **Compare Models**: Run analysis on different checkpoints to compare performance
3. **Deep Dive**: Focus on specific emotion classes or challenging samples
4. **Integration**: Use these insights to improve model training and deployment

### Key Takeaways:
- R-AIME provides comprehensive interpretability for multimodal emotion recognition
- Confidence calibration improves prediction reliability
- Cross-modal attention reveals text-audio interaction patterns
- Feature attribution helps understand model decision-making

For more advanced analysis, refer to the `raime_analysis.py` script.